In [3]:
# =============================================================================
# DEBUG_IV_CORE.IPYNB — CORRECT FIRST-STAGE (matches Saadaoui)
# =============================================================================

import pandas as pd
import numpy as np
from pathlib import Path
import pyreadstat
from statsmodels.api import OLS, add_constant
import warnings
warnings.filterwarnings('ignore')

# ----------------------------- PATHS -----------------------------
PROJ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DTA_PATH = PROJ / "data" / "Saadaoui_2026_JCE.dta"

print(f"Project root : {PROJ}\n")

# ----------------------------- LOAD & PREPARE -----------------------------
df_dta, _ = pyreadstat.read_dta(DTA_PATH)
base = pd.Timestamp('1960-01-01')
df_dta['date'] = df_dta['Period'].apply(lambda m: base + pd.DateOffset(months=int(m)))
df = df_dta.set_index('date').sort_index()
df.index = df.index.to_period('M').to_timestamp()

df['dllgop']  = df['llgop'].diff()
df['dl2lgop'] = df['l2lgop'].diff()

print(f"Date range: {df.index[0].date()} — {df.index[-1].date()} ({len(df)} months)\n")

# ----------------------------- BASELINE + LAGS OF TREATMENT -----------------------------
baseline = ['llwip', 'dllgop', 'l2lwip', 'dl2lgop']

# Add lags of lpri (this is what makes F-stat strong)
for lag in [1, 2]:
    df[f'L{lag}_lpri'] = df['lpri'].shift(lag)

controls = baseline + ['L1_lpri', 'L2_lpri']

# ----------------------------- FIRST STAGE -----------------------------
df_fs = df[['lpri', 'd2pri'] + controls].dropna()

X = add_constant(df_fs[['d2pri'] + controls])
y = df_fs['lpri']

model = OLS(y, X).fit()

# F-test on d2pri
z_idx = list(X.columns).index('d2pri')
r = np.zeros((1, len(X.columns)))
r[0, z_idx] = 1
f_test = model.f_test(r)

print("="*70)
print("CORRECT FIRST-STAGE (Saadaoui style)")
print("="*70)
print(f"F-statistic for d2pri : {f_test.fvalue:.3f}")
print(f"p-value               : {f_test.pvalue:.2e}")
print(f"Observations          : {len(df_fs)}")
print(f"R²                    : {model.rsquared:.4f}")
print(f"d2pri coefficient     : {model.params['d2pri']:.5f} "
      f"(SE = {model.bse['d2pri']:.5f})")
print("="*70)

if f_test.fvalue > 150:
    print("STRONG INSTRUMENT — Matches Saadaoui")
else:
    print("Still weak — we need to investigate further")

Project root : c:\Users\HP\Desktop\replication+contribution

Date range: 1990-01-01 — 2022-02-01 (386 months)

CORRECT FIRST-STAGE (Saadaoui style)
F-statistic for d2pri : 1795.267
p-value               : 3.08e-145
Observations          : 384
R²                    : 0.9968
d2pri coefficient     : 0.61119 (SE = 0.01442)
STRONG INSTRUMENT — Matches Saadaoui
